# Envanter hakem koşusu — DÜZ yön (Kaggle) · V4EC

Colab **ters** yönden, bu defter **düz** yönden koşar; ortada buluşurlar.

| | Colab | Kaggle (bu defter) |
|---|---|---|
| girdi | `needs_review_subset_reversed.csv` | `needs_review_subset.csv` |
| yön | son satırdan başa | ilk satırdan sona |
| çıktı | `inventory_v4ec.csv` | `kaggle_inventory_v4ec_forward.csv` |

## Çakışma bu sefer sorun değil

Geçen sefer Kaggle bölünmemiş dosyayı koşmuştu ve birleştirme varsayımı
çökmüştü. Bu sefer **ayrık dilim varsaymıyoruz**: `scripts/merge_runs.py`
`query` üzerinden tekilleştiriyor. İki koşu ortada üst üste binerse tek
bedeli boşa harcanan hesap olur, veri bozulmaz.

İkisi de aynı yapılandırmayı (v4 + `chosen` kapısı) kullandığı için hangi
kopyanın tutulduğu fark etmez.

## ⚠️ Eski çıktıdan devam ETME

`kaggle-judge-output` dataset'i **v1 prompt + kapısız** üretildi. Bu koşu
onu devam ettirmez, **yeni bir dataset'e** yazar. Eski satırlar atılıyor.

## Kota

Kaggle haftada 30 saat GPU veriyor. T4'te ~1.200 sorgu/saat → **~36.000
sorgu/hafta**. Yani Kaggle işi bitirmez, Colab'ın yükünü hafifletir.


## 1) GPU — P100 kabul edilmiyor


In [ ]:
!nvidia-smi
import subprocess
gpu = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print('GPU:', gpu or '(YOK)')
if 'P100' in gpu:
    raise RuntimeError('P100 secili - sm_60, PyTorch desteklemiyor, embedding cokuyor. '
                       "Session options -> Accelerator -> 'GPU T4 x2'")
assert gpu, "GPU yok - Session options -> Accelerator -> 'GPU T4 x2'"


## 2) Dataset yolları

Sağ panel → **Add Input** ile şu üçü ekli olmalı:
`gemma4-e4b-ollama` · `needs-review-full` · `institution-catalog-canonical`

Çıktı dataset'ini **elle oluşturmana gerek yok** — 11. hücre yoksa kendisi kurar.
Eski `kaggle-judge-output` kullanılmayacak (v1 prompt + kapısız üretildi).


In [ ]:
import os

# Kaggle'in gercek mount yolu '/kaggle/input/<slug>' DEGIL,
# '/kaggle/input/datasets/<kullanici>/<slug>' (2026-08-13 canli dogrulama).
KULLANICI = 'mcangultekin'
MODEL_DS  = f'/kaggle/input/datasets/{KULLANICI}/gemma4-e4b-ollama'
KATALOG_DS= f'/kaggle/input/datasets/{KULLANICI}/institution-catalog-canonical'
GIRDI     = f'/kaggle/input/datasets/{KULLANICI}/needs-review-full/needs_review_subset.csv'
CIKTI_DS  = 'inventory-v4ec-forward'   # YENI dataset - eskisi v1'di

for p in (MODEL_DS, KATALOG_DS):
    assert os.path.isdir(p), f"dataset yok: {p} - 'Add Input' ile eklediniz mi?"
assert os.path.isfile(GIRDI), f'girdi yok: {GIRDI}'

# DUZ dosya oldugunu dogrula - yanlislikla reversed eklenirse iki kosu
# ayni ucu tarar ve Kaggle kotasi bosa gider.
import csv
csv.field_size_limit(10_000_000)
with open(GIRDI, newline='', encoding='utf-8') as f:
    ilk = next(csv.DictReader(f))['query']
print('ilk sorgu:', ilk)
assert 'Azeebaycan' not in ilk, 'Bu REVERSED dosya! Colab ile ayni ucu tararsiniz.'


## 3) Kod


In [ ]:
import os
REPO = '/kaggle/working/institution_resolver_v3'
if not os.path.isdir(REPO):
    !git clone --branch feat/gate-asama1 https://github.com/mcangultekin/institution_resolver_v3.git {REPO}
else:
    !cd {REPO} && git fetch origin && git checkout feat/gate-asama1 && git pull
os.chdir(REPO)
!git log --oneline -1
assert os.path.isfile('src/institution_resolver_v3/retrieve/token_df.py'), 'kapi kodu yok - git pull?'


## 4) Katalog dosyaları


In [ ]:
import os
os.chdir(REPO); os.makedirs('data/processed', exist_ok=True)

def _link(kaynak, ad):
    hedef = f'data/processed/{ad}'
    if not os.path.exists(hedef):
        os.symlink(f'{kaynak}/{ad}', hedef)

for ad in os.listdir(KATALOG_DS):
    _link(KATALOG_DS, ad)

# embeddings.npz katalog dataset'inde YOK (2026-08-15 dogrulandi: sadece iki
# JSONL var). Yoksa indeksleme 231 bin kaydi sifirdan encode ediyor. Ayri bir
# dataset olarak yuklenirse burasi onu kendiliginden bulur - defteri tekrar
# degistirmek gerekmez, sadece 'Add Input' ile eklemek yeterli.
EMBED_DS = f'/kaggle/input/datasets/{KULLANICI}/institution-embeddings'
if os.path.isdir(EMBED_DS):
    for ad in os.listdir(EMBED_DS):
        _link(EMBED_DS, ad)

!ls -la data/processed
if os.path.exists('data/processed/embeddings.npz'):
    print('\nembeddings.npz VAR - indeksleme encode etmeyecek')
else:
    print('\n*** embeddings.npz YOK ***')
    print('    Ilk indeksleme 231.291 kaydi encode edecek (T4"te ~20-30 dk).')
    print("    Kalicilastirmak icin: embeddings.npz'yi 'institution-embeddings'")
    print("    adiyla dataset olarak yukleyip 'Add Input' ile ekleyin.")


## 5) Elasticsearch

Oturum yenilendiyse Kaggle persistence `jdk/bin/java`'nın çalıştırma bitini
siliyor ve ES `Permission denied` veriyor. Aşağıdaki hücre bunu kendi
başına onarır — arşivi yeniden açar ama `data/` dizinini korur.


In [ ]:
%%bash
set -e
# KOK NEDEN (2026-08-15): ES ikilileri /kaggle/working'deydi ve Kaggle
# persistence geri yuklerken calistirma bitlerini soyuyor -> jdk/bin/java
# "Permission denied". chmod ile onarmak kirilgan cikti: arsiv da persistence'ta
# kalmayabiliyor, o zaman `tar` patliyor ve `set -e` chmod'a hic sira gelmeden
# script'i kesiyor (gorulen log onceki denemeden kaliyor, teshis yaniltiyor).
#
# COZUM: IKILILER kalici OLMAYAN /opt/es'te (her oturum taze, bit sorunu yok),
# VERI kalici /kaggle/working/es_data'da (indeks oturumlar arasi yasiyor).
ES=/opt/es
VERI=/kaggle/working/es_data
TAR=/kaggle/working/es.tar.gz

[ -f $TAR ] || wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-8.14.0-linux-x86_64.tar.gz -O $TAR
rm -rf $ES && mkdir -p $ES $VERI
tar -xzf $TAR -C $ES --strip-components=1

cat > $ES/config/elasticsearch.yml <<EOF
discovery.type: single-node
xpack.security.enabled: false
xpack.security.http.ssl.enabled: false
xpack.ml.enabled: false
path.data: $VERI
EOF

cat > $ES/config/elasticsearch.policy <<'POLEOF'
grant {
  permission java.io.FilePermission "/sys/-", "read";
  permission java.io.FilePermission "/proc/-", "read";
};
POLEOF

sysctl -w vm.max_map_count=262144 || true
id -u esuser &>/dev/null || useradd -m esuser
chown -R esuser:esuser $ES $VERI

# java gercekten calisiyor mu - ES'i baslatmadan ONCE dogrula ki hata
# 90 saniyelik saglik dongusunun sonunda degil, burada gorunsun.
sudo -u esuser $ES/jdk/bin/java -version 2>&1 | head -1

pkill -f 'org.elasticsearch.bootstrap.Elasticsearch' 2>/dev/null || true
sleep 1
sudo -u esuser env ES_JAVA_OPTS="-Xms4g -Xmx4g -Djava.security.policy=$ES/config/elasticsearch.policy" \
  setsid $ES/bin/elasticsearch < /dev/null > /kaggle/working/es.log 2>&1 &
disown
sleep 3


In [ ]:
import time, requests
for _ in range(90):
    try:
        r = requests.get('http://localhost:9200/_cluster/health', timeout=5)
        if r.status_code == 200:
            print('ES:', r.json()['status']); break
    except requests.exceptions.RequestException:
        pass
    time.sleep(2)
else:
    !tail -30 /kaggle/working/es.log
    raise RuntimeError('ES baslatilamadi')


## 6) Paketler


In [ ]:
import os; os.chdir(REPO)
!pip uninstall -y torchaudio torchvision 2>/dev/null || true
!pip install -q --force-reinstall -e '.[dev,embed,llm,api]'


## 7) Ollama

T4 **15 GB**, model **9,61 GB**. `NUM_PARALLEL` her slot için ayrı KV ayırdığından
burada **4** slot ayrılır, koşu ise **2 işçi** ile döner. 8 denerseniz katmanlar
CPU'ya taşar ve hız çöker. (Colab L4/A100'de 8 kullanılıyor; Kaggle'ın kısıtı farklı.)


In [ ]:
%%bash
set -e
apt-get update -qq && apt-get install -y -qq zstd
command -v ollama >/dev/null || curl -fsSL https://ollama.com/download/ollama-linux-amd64.tar.zst | tar --zstd -x -C /usr
ollama --version


In [ ]:
import os, subprocess, time, requests

os.environ['OLLAMA_MODELS']       = MODEL_DS   # salt-okunur dataset, pull gerekmiyor
os.environ['OLLAMA_KEEP_ALIVE']   = '-1'
os.environ['OLLAMA_NUM_PARALLEL'] = '4'  # kosu 2 isciyle; 4 slot ki hiz testi 4'u de olcebilsin

try:
    ayakta = requests.get('http://localhost:11434/api/tags', timeout=5).status_code == 200
except requests.exceptions.RequestException:
    ayakta = False
if not ayakta:
    subprocess.Popen(['ollama','serve'], stdout=open('/kaggle/working/ollama.log','w'),
                     stderr=subprocess.STDOUT, env=os.environ)

names = None
for _ in range(40):
    try:
        # timeout=30: dataset uzerinden model taramasi yavas, 2 sn yetmiyor.
        r = requests.get('http://localhost:11434/api/tags', timeout=30)
        if r.status_code == 200:
            names = [m['name'] for m in r.json().get('models', [])]; break
    except requests.exceptions.RequestException:
        pass
    time.sleep(3)
if names is None:
    print(open('/kaggle/working/ollama.log').read()[-3000:]); raise RuntimeError('Ollama baslamadi')
print('Ollama:', names)
assert any('gemma4' in n for n in names), f'gemma4:e4b yok: {names}'

r = requests.post('http://localhost:11434/api/generate',
                  json={'model':'gemma4:e4b','prompt':'Merhaba','stream':False}, timeout=600)
print('warm-up:', 'OK' if r.status_code == 200 else r.text[:200])


## 8) İndeksleme


In [ ]:
import os, sys, subprocess, requests
os.chdir(REPO)

# Indeks zaten doluysa atla. Kaggle/Colab persistence ES verisini saklayabiliyor
# ve `setup-es` indeksi yeniden kurdugu icin embedding'ler de bosuna yeniden
# uretiliyordu. Kaggle'da embeddings.npz DATASET'TE YOK - orada bu adim 231 bin
# kaydi sifirdan encode ediyor (~20-30 dk), her oturum basi.
def _sayi():
    try:
        r = requests.get('http://localhost:9200/institutions_v1/_count', timeout=30)
        return r.json()['count'] if r.status_code == 200 else 0
    except Exception:
        return 0

n = _sayi()
if n == 231291:
    print(f'indeks zaten dolu ({n}) - setup-es/index atlaniyor')
else:
    if n:
        print(f'indekste {n} kayit var (eksik) - yeniden kuruluyor')
    for arg in (['setup-es'], ['index', '--embeddings']):
        p = subprocess.run([sys.executable, '-m', 'institution_resolver_v3.cli.main', *arg])
        if p.returncode != 0:
            raise RuntimeError(f'{arg[0]} hata verdi (kod {p.returncode})')
    n = _sayi()
print('indekslenen kayit:', n)
assert n == 231291, f'beklenen 231291, gelen {n}'


## 9) Duman testi — v4 ve kapı devrede mi

Colab defteriyle **aynı** üç sorgu. İki koşunun aynı yapılandırmada olduğunu
burada doğruluyoruz; birleştirmenin geçerliliği buna bağlı.


In [ ]:
import csv, os
os.chdir(REPO); os.makedirs('/kaggle/working/tmp', exist_ok=True)
with open('/kaggle/working/tmp/duman.csv','w',newline='',encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['query','normalized_name','rows']); w.writeheader()
    for q in ['University of South Australia','AFAD','T.C. Ticaret Bakanlığı']:
        w.writerow({'query':q,'normalized_name':q.lower(),'rows':'1'})

!python3 -m institution_resolver_v3.cli.main inventory-batch /kaggle/working/tmp/duman.csv --out /kaggle/working/tmp/duman_out.csv

csv.field_size_limit(10_000_000)
rows = list(csv.DictReader(open('/kaggle/working/tmp/duman_out.csv', encoding='utf-8')))
for r in rows:
    print(f"{r['query'][:30]:<32} prompt={r['prompt_variant']:<3} kapi={r['pool_gate']:<8} "
          f"ates={r['gate_orphan_fired'] or '-':<3} {r['parent_verdict']:<11} {r['parent_name'][:28]:<30} {r['orphan_tokens']}")
assert all(r['prompt_variant'] == 'v4' for r in rows), 'v4 devrede degil'
d = {r['query']: r for r in rows}
# AFAD kapinin SESSIZ kalmasi gereken vaka (kimlik token'i alias'ta geciyor).
# Hakem hata verirse kapi hic calismaz ve alan bos kalir - bu kapinin yanlisi
# degil, ayri bir olay; oyle durumda uyar ama testi dusurme.
afad = d['AFAD']
if afad['status'] != 'ok':
    print(f"\nUYARI: AFAD hakemde hata aldi -> {afad['error'][:80]}")
    print('  Kapi bu satirda hic calismadi; kapi dogrulamasi yapilamadi.')
else:
    assert afad['gate_orphan_fired'] == '0', 'kapi AFAD uzerinde yanlis ateşledi'
assert d['University of South Australia']['gate_orphan_fired'] == '1', 'kapi ateşlemeliydi'
print('\nDuman testi: v4 + kapi devrede.')


## 10) İşçi hız testi

T4 15 GB'da model 9,61 GB yer kapliyor; `NUM_PARALLEL=4` slot ayrildi, o yuzden 1/2/4.


In [ ]:
import os, csv, time
os.chdir(REPO)
sonuc = []
for w in (1, 2, 4):
    out = f'/kaggle/working/tmp/w{w}.csv'
    if os.path.exists(out): os.remove(out)
    t0 = time.time()
    !python3 -m institution_resolver_v3.cli.main inventory-batch '{GIRDI}' --out {out} --limit 30 --workers {w} > /kaggle/working/tmp/w{w}.log 2>&1
    dt = time.time() - t0
    n = sum(1 for _ in csv.DictReader(open(out, encoding='utf-8'))) if os.path.exists(out) else 0
    sonuc.append((w, dt, n))
    print(f'workers={w}  {dt:6.1f} sn  {n} satir  {dt/max(n,1):5.2f} sn/sorgu')

print()
for w, dt, n in sonuc:
    sq = dt/max(n,1)
    print(f'workers={w}  {3600/sq:5.0f} sorgu/saat  ->  30 saatlik kotada ~{30*3600/sq:,.0f} sorgu')


## 11) Yedekleme


In [ ]:
import os, json, subprocess, shutil, tempfile, time

def _token():
    from kaggle_secrets import UserSecretsClient
    t = UserSecretsClient().get_secret('KAGGLE_API_TOKEN')
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    p = os.path.expanduser('~/.kaggle/access_token')
    open(p,'w').write(t.strip()); os.chmod(p, 0o600)
_token()

def yedekle(csv_path, slug=None, mesaj=None):
    """csv'yi Kaggle Dataset'ine push eder; dataset YOKSA olusturur.

    Kaggle arayuzu bos dataset olusturmaya izin vermiyor (en az bir dosya
    istiyor), o yuzden ilk yedek `create`, sonrakiler `version` ile gider.
    Elle on hazirlik gerekmez.
    """
    slug = slug or CIKTI_DS
    st = tempfile.mkdtemp()
    shutil.copy(csv_path, os.path.join(st, os.path.basename(csv_path)))
    json.dump({'title': slug, 'id': f'{KULLANICI}/{slug}',
               'licenses': [{'name':'CC0-1.0'}]},
              open(os.path.join(st,'dataset-metadata.json'),'w'))

    r = subprocess.run(['kaggle','datasets','version','-p',st,'-m',
                        mesaj or f'auto {time.strftime("%H:%M")}','--dir-mode','zip'],
                       capture_output=True, text=True)
    cikti = (r.stdout or '') + (r.stderr or '')
    if r.returncode != 0 and ('not found' in cikti.lower() or '404' in cikti):
        print(f'{slug} yok, olusturuluyor...')
        r = subprocess.run(['kaggle','datasets','create','-p',st,'--dir-mode','zip'],
                           capture_output=True, text=True)
        cikti = (r.stdout or '') + (r.stderr or '')
    if r.returncode != 0:
        print('YEDEK BASARISIZ:', cikti[-500:])
    return r.returncode == 0

# Ilk yedegi hemen dene - kosu 12 saat surup sonunda token hatasi almaktansa
# simdi ogrenelim. Kucuk bir yer tutucu dosyayla.
open('/kaggle/working/_ilk.csv','w').write('query\n_baslangic\n')
print('yedekleme testi:', 'OK' if yedekle('/kaggle/working/_ilk.csv', mesaj='init') else 'BASARISIZ')


## 12) KOŞU

Kaggle oturumu 12 saatte kapanır. Kesilirse: `inventory-v4ec-forward`
dataset'ini **Add Input** ile ekle, aşağıdaki `ONCEKI` satırını aç, tekrar koş.


In [ ]:
import os, csv, time, shutil, sys, subprocess, requests
from IPython.display import clear_output
os.chdir(REPO)

WORKERS = 2                # Kaggle T4 varsayilani
CHUNK   = 500
YEREL   = '/kaggle/working/kaggle_inventory_v4ec_forward.csv'
HEDEF   = 143039           # Colab ortada bulusunca bu sayiya varmadan durdurulur
csv.field_size_limit(10_000_000)

# Onceki oturumdan devam - OTOMATIK. Kaggle oturumu 12 saatte kapaniyor;
# her parcada CIKTI_DS'e yedekliyoruz, yeni oturum onu geri yukluyor.
# Tek sart: sag panel -> Add Input ile 'inventory-v4ec-forward' ekli olmali.
ONCEKI = f'/kaggle/input/datasets/{KULLANICI}/{CIKTI_DS}/kaggle_inventory_v4ec_forward.csv'
if os.path.exists(ONCEKI) and not os.path.exists(YEREL):
    shutil.copy(ONCEKI, YEREL)
    print(f'onceki yedek geri yuklendi: {ONCEKI}')
elif not os.path.exists(YEREL):
    print("UYARI: onceki yedek yok, SIFIRDAN baslanacak.")
    print("  Devam etmek istiyorsaniz 'inventory-v4ec-forward' dataset'ini")
    print('  Add Input ile ekleyip bu hucreyi tekrar calistirin.')

# Gecici = servis/ag kaynakli, yeniden denemek anlamli.
# Kalici = modelin cevabi (JudgeValidationError, ~%7 - v1'de de ayniydi):
# tekrar denemek AYNI sonucu verir, sadece kota yakar ve sonunda her parca
# tamamen tekrarlardan olusup dongu kilitlenir.
# resume duruma bakmadan sayiyor (csv_runner.py:63), o yuzden hangi satirin
# kalacagina burada karar vermek zorundayiz - kalan satir bir daha denenmez.
GECICI = ('LlmError', 'ConnectError', 'ReadTimeout', 'Timeout', 'Errno 111')

def _temizle(p):
    if not os.path.exists(p):
        return 0, 0
    with open(p, newline='', encoding='utf-8') as f:
        rd = csv.DictReader(f)
        alanlar = rd.fieldnames
        rows = list(rd)
    tut = [r for r in rows
           if r['status'] == 'ok' or not any(g in r['error'] for g in GECICI)]
    if len(tut) != len(rows):
        with open(p, 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=alanlar)
            w.writeheader()
            w.writerows(tut)
    return len(tut), len(rows) - len(tut)

def _ollama_ayakta():
    try:
        return requests.get('http://localhost:11434/api/tags', timeout=30).status_code == 200
    except requests.exceptions.RequestException:
        return False

def _ollama_kaldir():
    subprocess.Popen(['ollama', 'serve'], stdout=open('/kaggle/working/ollama.log', 'a'),
                     stderr=subprocess.STDOUT, env=os.environ)
    for _ in range(40):
        if _ollama_ayakta():
            return True
        time.sleep(3)
    return False

ozet = []
baslangic = time.time()

while True:
    onceki, atilan = _temizle(YEREL)
    if onceki >= HEDEF:
        print('TAMAMLANDI:', onceki)
        break

    if not _ollama_ayakta():
        ozet.append('  ! Ollama olmustu, yeniden baslatildi')
        if not _ollama_kaldir():
            raise RuntimeError('Ollama ayaga kalkmadi - !tail -40 /kaggle/working/ollama.log')

    # Ekrani her parcada temizle: 286 parca x 500 satir birikirse tarayici donar.
    clear_output(wait=True)
    for s in ozet[-15:]:
        print(s)
    print(f'--- parca {len(ozet)+1}: {onceki:,}/{HEDEF:,} '
          f'{"(" + str(atilan) + " gecici hata yeniden denenecek)" if atilan else ""}')

    t0 = time.time()
    p = subprocess.Popen(
        [sys.executable, '-u', '-m', 'institution_resolver_v3.cli.main',
         'inventory-batch', GIRDI, '--out', YEREL,
         '--workers', str(WORKERS), '--limit', str(CHUNK), '--resume'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for satir in p.stdout:
        print(satir, end='')
    p.wait()

    simdi, _ = _temizle(YEREL)
    yedekle(YEREL, mesaj=f'{simdi} satir')

    dt = time.time() - t0
    yeni = simdi - onceki
    if yeni <= 0:
        print('ILERLEME YOK - duruluyor')
        break
    kalan_sa = (HEDEF - simdi) * (time.time() - baslangic) / simdi / 3600
    ozet.append(f'  {simdi:,}/{HEDEF:,} (+{yeni})  {dt/60:.1f} dk  '
                f'{dt/yeni:.2f} sn/sorgu  ~{kalan_sa:.0f} sa kaldi  {time.strftime("%H:%M")}')


## 13) Özet + son yedek


In [ ]:
import csv, collections
csv.field_size_limit(10_000_000)
rows = list(csv.DictReader(open(YEREL, encoding='utf-8')))
print(f'{len(rows)} satir | ilk: {rows[0]["query"][:40]} | son: {rows[-1]["query"][:40]}\n')
print('parent karari:', dict(collections.Counter(r['parent_verdict'] or '(bos)' for r in rows)))
ates = sum(1 for r in rows if r['gate_orphan_fired'] == '1')
print(f'kapi ateşleyen: {ates} ({100*ates/max(len(rows),1):.1f}%)')
print('durum        :', dict(collections.Counter(r['status'] for r in rows)))
yedekle(YEREL, mesaj=f'final {len(rows)} satir')
print('\nSon hali yedeklendi. Colab ciktisiyla birlestirmek icin:')
print('  python3 scripts/merge_runs.py <colab.csv> <kaggle.csv> --out birlesik.csv')
